# Multimodal event logs: does text add evidence?

Compare structured payload, free-text payload, and a combined representation on a temporal holdout. An additional modality is useful only when it adds valid and stable evidence.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
while not (root / 'labs').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
from labs.common import load_module, load_variant_lab
lab = load_module('multimodal_lab', root / 'labs/07-multimodal-event-logs/multimodal_lab.py')
events = load_variant_lab().generate_event_log()
cases = lab.multimodal_case_table(events)
metrics, predictions = lab.evaluate_modalities(cases)
display(cases[['case_id', 'customer_note', 'channel', 'product_type', 'target']].head(12))
display(metrics)

,case_id,customer_note,channel,product_type,target
0,C-0001,standard request,web,standard,1
1,C-0002,standard request,web,standard,0
2,C-0003,,web,custom,1
3,C-0004,standard request,web,standard,0
4,C-0005,standard request,web,standard,0
5,C-0006,standard request,web,standard,0
6,C-0007,service activation request,web,service,0
7,C-0008,,direct,NaN,0
8,C-0009,,web,standard,1
9,C-0010,standard request,partner,standard,0


,modality,roc_auc,brier,auc_gain_over_structured
0,structured,0.475397,0.279717,0.000000
1,text,0.569444,0.252734,0.094048
2,combined,0.527778,0.273060,0.052381


In [2]:
display(predictions.head())
display(cases.groupby(cases['customer_note'].eq('').rename('missing_text'))['target'].agg(['size', 'mean']))

,case_id,target,start_time,structured_risk,text_risk,combined_risk
168,C-0169,1,2025-05-12 11:44:19.306670510,0.405792,0.453120,0.399089
169,C-0170,1,2025-05-13 06:07:14.887188636,0.314512,0.383195,0.246637
170,C-0171,0,2025-05-13 21:10:47.394877350,0.446820,0.453120,0.482036
171,C-0172,0,2025-05-14 15:57:36.638900714,0.802179,0.694363,0.828293
172,C-0173,1,2025-05-15 10:07:08.028165998,0.600775,0.453120,0.617476


,size,mean
missing_text,,
False,176,0.409091
True,64,0.218750


## Modality review

Does text improve the temporal holdout result enough to justify extraction, storage, privacy, and drift-monitoring costs? Discuss document timing, embeddings, sensitive information, missing modalities, and provenance.